<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-04-rag/lesson-4.2-diy-rag/practice/GCP_Capstone_4.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 4.2 — DIY RAG Pipeline

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Install dependencies, authenticate with Application Default Credentials, and initialize the Vertex `google-genai` client plus the Firestore client. Run this cell first — every exercise below depends on the names it defines (`client`, `db`, `types`, the Pydantic models, and the vector helpers).

> Set `PROJECT_ID` to your own project. Region is `us-central1` for the course examples (use `asia-south1` for India production).

In [ ]:
!pip install -q google-genai google-cloud-firestore pydantic

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google import genai
from google.genai import types
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from pydantic import BaseModel, Field
from typing import List, Literal
import json

USD_INR = 85  # cost display conversion

client = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')  # embeddings: regional only
gen_client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')   # Gemini 3.x generation: global only
db = firestore.Client(project=PROJECT_ID)
print('Client + Firestore ready')

## Exercise 1: Embed + Retrieve

**Difficulty:** Easy

Embed a query with RETRIEVAL_QUERY. Retrieve top-5 from Firestore. Print similarities.

1. Use embed_query() with task_type=RETRIEVAL_QUERY
2. Call retrieve_chunks() with distance_threshold=0.3
3. Print chunk_id, source, and similarity for each

In [ ]:
# Stage 1 — embed the query with RETRIEVAL_QUERY (asymmetric task type for the question side)
def embed_query(query):
    result = client.models.embed_content(
        model='text-embedding-005', contents=query,
        config=types.EmbedContentConfig(
            task_type='RETRIEVAL_QUERY',
            output_dimensionality=768))
    return result.embeddings[0].values

# Stage 2 — retrieve nearest chunks from Firestore (COSINE + distance_threshold quality gate)
def retrieve_chunks(query_vector, collection='rag_chunks', top_k=5, distance_threshold=0.3):
    results = db.collection(collection).find_nearest(
        vector_field='embedding', query_vector=Vector(query_vector),
        distance_measure=DistanceMeasure.COSINE,
        limit=top_k, distance_threshold=distance_threshold,
        distance_result_field='vector_distance').get()
    chunks = []
    for doc in results:
        data = doc.to_dict()
        chunks.append({
            'chunk_id': doc.id,
            'content': data.get('content', ''),
            'source': data.get('source_file', 'unknown'),
            'pages': data.get('pages', ''),
            'distance': data.get('vector_distance'),
        })
    return chunks

qv = embed_query('What are the latest trends in RAG?')
print(f'Query vector: {len(qv)} dims, first 5: {qv[:5]}')

chunks = retrieve_chunks(qv)
print(f'\nRetrieved {len(chunks)} chunks')
for c in chunks:
    # COSINE distance -> similarity ~ 1 - distance (lower distance = more relevant)
    d = c.get('distance')
    sim = (1 - d) if d is not None else 'n/a'
    print(f'  {c["chunk_id"]} | {c["source"]} | similarity={sim}')

## Exercise 2: Build Augmented Prompt

**Difficulty:** Easy

Build a numbered context prompt from retrieved chunks. Print the full prompt.

1. Number each chunk with [Source N]
2. Include metadata (source, pages)
3. Add citation instructions

In [ ]:
# Stage 3 — augment: turn retrieved chunks into a numbered [Source N] context block
def build_rag_prompt(query, chunks):
    if not chunks:
        return f'Question: {query}\n\nNo relevant context found.'
    ctx = '\n\n'.join(
        f'[Source {i+1}] ({c["source"]}, pages {c["pages"]})\n{c["content"]}'
        for i, c in enumerate(chunks))
    instructions = 'Answer ONLY from the context above. Cite every claim with [Source N].'
    return f'Context:\n{ctx}\n\n{instructions}\n\nQuestion: {query}'

prompt = build_rag_prompt('What are trends in RAG?', chunks)
print(prompt[:400])

## Exercise 3: Cost Calculator

**Difficulty:** Easy

Calculate per-query cost from usage_metadata. Print USD and INR breakdown.

1. Extract prompt_token_count, candidates_token_count, thoughts_token_count
2. Apply Flash pricing: $1.50/M input, $7.50/M output
3. Convert to INR

> Convert at the current course standard `USD_INR = 85`.

In [ ]:
# Stage 5 — cost tracking from usage_metadata (gemini-3.6-flash standard: $1.50 in / $7.50 out per 1M)
def calc_cost(usage, n_chunks=5):
    inp = usage.prompt_token_count
    out = usage.candidates_token_count
    think = usage.thoughts_token_count or 0
    cost = inp / 1e6 * 1.50 + (out + think) / 1e6 * 7.50 + 0.00001  # + tiny embed cost
    print(f'Input:    {inp} tokens (${inp/1e6*1.50:.6f})')
    print(f'Output:   {out} tokens (${out/1e6*7.50:.6f})')
    print(f'Thinking: {think} tokens')
    print(f'Total:    ${cost:.6f} (Rs {cost*USD_INR:.4f})')
    return cost

# Needs `usage` from Exercise 4; run this after generating once.
try:
    calc_cost(usage)
except NameError:
    print('Run Exercise 4 first to produce `usage`, then re-run this cell.')

## Exercise 4: RAGResponse with Pydantic

**Difficulty:** Medium

Generate structured RAG answer with citations. Verify typed response.parsed.

1. Define Citation + RAGResponse Pydantic models
2. Generate with response_schema=RAGResponse
3. Access typed fields: result.answer, result.citations

In [ ]:
class Citation(BaseModel):
    source_id: int = Field(description='Source number [1-N]')
    chunk_id: str = Field(description='Firestore doc ID')
    relevance: float

class RAGResponse(BaseModel):
    answer: str = Field(description='Answer from context only')
    confidence: Literal['high', 'medium', 'low']
    citations: List[Citation]
    needs_more_context: bool

RAG_SYSTEM = '''You are DocuMind. Answer ONLY from context.
Cite every claim with [Source N]. Rate confidence.'''

# Stage 4 — generate structured, cited answer. thinking_budget=0 keeps latency/cost low for extraction.
def generate_rag(prompt, chunks):
    chunk_map = {i+1: c['chunk_id'] for i, c in enumerate(chunks)}
    r = gen_client.models.generate_content(
        model='gemini-3.6-flash', contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=RAG_SYSTEM,
            response_mime_type='application/json',
            response_schema=RAGResponse,
            temperature=0.1,
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    result = r.parsed  # typed RAGResponse
    # backfill real Firestore doc IDs from the [Source N] map
    for c in result.citations:
        c.chunk_id = chunk_map.get(c.source_id, 'unknown')
    return result, r.usage_metadata

result, usage = generate_rag(prompt, chunks)
print(f'Answer: {result.answer[:200]}...')
print(f'Confidence: {result.confidence}')
print(f'Citations: {len(result.citations)}')

## Exercise 5: Quality Gate Test

**Difficulty:** Medium

Test distance_threshold at 0.1 (strict), 0.3 (balanced), 0.8 (loose). Compare results.

1. Run same query with 3 different thresholds
2. Count chunks returned at each level
3. Observe: strict returns fewer but more relevant chunks

In [ ]:
# Sweep the distance_threshold parameter of retrieve_chunks() to see the quality gate in action.
for thr in (0.1, 0.3, 0.8):
    hits = retrieve_chunks(qv, top_k=5, distance_threshold=thr)
    print(f'threshold={thr}: {len(hits)} chunks -> ' +
          ', '.join(h['source'] for h in hits))

print('\nStrict (0.1) returns fewer but more relevant chunks; loose (0.8) lets low-quality matches through.')

## Exercise 6: Citation Verification

**Difficulty:** Medium

Build verify_citations(). Flag invalid source_ids and uncited claims.

1. Check source_ids are within valid range
2. Count sentences without [Source N] citations
3. Report issues list

In [ ]:
def verify_citations(rag_result, chunks):
    valid_ids = set(range(1, len(chunks) + 1))
    issues = []
    for c in rag_result.citations:
        if c.source_id not in valid_ids:
            issues.append(f'Invalid source_id: {c.source_id}')
    sentences = rag_result.answer.split('. ')
    uncited = sum(1 for s in sentences if '[Source' not in s)
    if uncited > 1:
        issues.append(f'{uncited} uncited sentences')
    return {'valid': len(issues) == 0, 'issues': issues}

v = verify_citations(result, chunks)
print(f'Valid: {v["valid"]}')
for issue in v['issues']:
    print(f'  Issue: {issue}')

## Exercise 7: Full ask_documind()

**Difficulty:** Challenge

Wire all 5 stages. Test with 5 diverse queries. Print answers + costs.

1. Combine embed + retrieve + augment + generate + cost
2. Handle empty chunks gracefully
3. Test with varied queries

In [ ]:
# Complete pipeline: embed -> retrieve -> augment -> generate -> cost
def ask_documind(query, top_k=5):
    qv = embed_query(query)
    chunks = retrieve_chunks(qv, top_k=top_k)
    prompt = build_rag_prompt(query, chunks)
    if not chunks:  # graceful empty-retrieval handling
        return {'answer': 'No relevant context.', 'confidence': 'low', 'citations': [], 'cost_usd': 0}
    result, usage = generate_rag(prompt, chunks)
    cost = usage.prompt_token_count / 1e6 * 1.50 + usage.candidates_token_count / 1e6 * 7.50
    return {
        'answer': result.answer,
        'confidence': result.confidence,
        'citations': [c.model_dump() for c in result.citations],
        'needs_more': result.needs_more_context,
        'cost_usd': cost}

queries = [
    'What is RAG?',
    'How does chunking affect retrieval quality?',
    'What embedding model does DocuMind use?',
    'How are citations tracked in the answer?',
    'What is the cost per query?',
]
for q in queries:
    r = ask_documind(q)
    print(f'Q: {q}')
    print(f'  Answer: {r["answer"][:120]}...')
    print(f'  Confidence: {r["confidence"]} | Citations: {len(r["citations"])} | '
          f'Cost: ${r["cost_usd"]:.6f} (Rs {r["cost_usd"]*USD_INR:.4f})')
    print()

## Exercise 8: RAGEngine Module

**Difficulty:** Challenge

Build complete RAGEngine class. Track stats across 10 queries. Print report.

1. Implement query() with all 5 stages
2. Track total_cost and query_count
3. Implement report() with averages

In [ ]:
class RAGEngine:
    def __init__(self, project, location='us-central1'):
        self.client = genai.Client(enterprise=True, project=project, location=location)      # embeddings: regional
        self.gen_client = genai.Client(enterprise=True, project=project, location='global')  # generation: global
        self.db = firestore.Client(project=project)
        self.total_cost = 0.0
        self.query_count = 0

    def query(self, question, collection='rag_chunks', top_k=5):
        qv = self.client.models.embed_content(
            model='text-embedding-005', contents=question,
            config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY',
                                            output_dimensionality=768)
        ).embeddings[0].values
        docs = self.db.collection(collection).find_nearest(
            vector_field='embedding', query_vector=Vector(qv),
            distance_measure=DistanceMeasure.COSINE,
            limit=top_k, distance_threshold=0.3).get()
        chunks = [{'id': d.id, 'content': d.to_dict().get('content', ''),
                   'source': d.to_dict().get('source_file', '')} for d in docs]
        if not chunks:
            return {'answer': 'No relevant context.', 'confidence': 'low'}
        ctx = '\n\n'.join(f'[Source {i+1}]\n{c["content"]}' for i, c in enumerate(chunks))
        r = self.gen_client.models.generate_content(
            model='gemini-3.6-flash', contents=f'Context:\n{ctx}\n\nQuestion: {question}',
            config=types.GenerateContentConfig(
                system_instruction='Answer from context only. Cite [Source N].',
                response_mime_type='application/json', response_schema=RAGResponse,
                temperature=0.1, thinking_config=types.ThinkingConfig(thinking_budget=0)))
        cost = r.usage_metadata.prompt_token_count / 1e6 * 1.50 + \
               r.usage_metadata.candidates_token_count / 1e6 * 7.50
        self.total_cost += cost
        self.query_count += 1
        return {'result': r.parsed, 'cost': cost, 'chunks': len(chunks)}

    def report(self):
        avg = self.total_cost / self.query_count if self.query_count else 0
        print(f'Queries: {self.query_count} | Total: ${self.total_cost:.4f} '
              f'(Rs {self.total_cost*USD_INR:.2f}) | Avg: ${avg:.6f}/q')

engine = RAGEngine(PROJECT_ID)
test_queries = [
    'What is RAG?', 'How does retrieval work?', 'What is chunking?',
    'How are embeddings generated?', 'What is a vector database?',
    'How does DocuMind cite sources?', 'What model powers generation?',
    'How is confidence rated?', 'What is the distance threshold for?',
    'How is per-query cost computed?',
]
for q in test_queries:
    engine.query(q)
engine.report()